<a href="https://colab.research.google.com/github/VintaBytes/Ciencia-de-datos/blob/main/libros/ciencia-de-datos-con-python/polars-01/cuadernos/Polars_Cuaderno07_Transformar_columnas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Cuaderno 7 - Crear y transformar columnas con `with_columns()`

## Antes de comenzar

En los cuadernos anteriores utilizamos expresiones para seleccionar columnas y para construir condiciones de filtrado. Ahora vamos a emplearlas con otro propósito central: **agregar nuevas columnas o reemplazar columnas existentes**.

La herramienta principal será `with_columns()`.

El objetivo de este cuaderno es observar con claridad la diferencia entre `select()` y `with_columns()`, crear una o varias columnas calculadas, reemplazar columnas, comprobar cómo pueden cambiar los tipos y encadenar transformaciones cuando una columna nueva depende de otra creada previamente.

Todavía no utilizaremos transformaciones condicionales con `when`, `then` y `otherwise`. Ese será el tema del próximo capítulo.


## 1. Preparar el entorno y los datos

Instalamos Polars en Colab, importamos la biblioteca y reconstruimos el DataFrame de ventas que venimos utilizando.


In [20]:
!pip install -q polars

In [21]:
import polars as pl

In [22]:
datos = {
    "producto": ["Cuaderno", "Lapicera", "Mochila", "Calculadora", "Regla", "Carpeta"],
    "categoria": ["Librería", "Librería", "Accesorios", "Tecnología", "Librería", "Librería"],
    "precio": [3500, 1200, 28500, 18500, 900, 4200],
    "cantidad": [4, 10, 2, 3, 6, 5],
}

df = pl.DataFrame(datos)

df

producto,categoria,precio,cantidad
str,str,i64,i64
"""Cuaderno""","""Librería""",3500,4
"""Lapicera""","""Librería""",1200,10
"""Mochila""","""Accesorios""",28500,2
"""Calculadora""","""Tecnología""",18500,3
"""Regla""","""Librería""",900,6
"""Carpeta""","""Librería""",4200,5


El DataFrame ejecutado tiene `shape: (6, 4)` y contiene las columnas `producto`, `categoria`, `precio` y `cantidad`. `precio` y `cantidad` son de tipo `Int64`. A partir de esta tabla vamos a construir nuevas variables sin modificar el objeto original.


## 2. Recordar qué hacía `select()`

Antes de introducir `with_columns()`, retomamos una operación conocida: calcular el importe dentro de `select()`.


In [23]:
df.select(
    "producto",
    (pl.col("precio") * pl.col("cantidad")).alias("importe"),
)

producto,importe
str,i64
"""Cuaderno""",14000
"""Lapicera""",12000
"""Mochila""",57000
"""Calculadora""",55500
"""Regla""",5400
"""Carpeta""",21000


La salida tiene `shape: (6, 2)` y contiene solamente `producto` e `importe`. Los importes calculados son `14000`, `12000`, `57000`, `55500`, `5400` y `21000`.

Este resultado recuerda el comportamiento de `select()`: la tabla final queda formada únicamente por las expresiones que indicamos.


## 3. Agregar una columna con `with_columns()`

Ahora realizamos el mismo cálculo dentro de `with_columns()`.


In [24]:
df.with_columns(
    (pl.col("precio") * pl.col("cantidad")).alias("importe")
)

producto,categoria,precio,cantidad,importe
str,str,i64,i64,i64
"""Cuaderno""","""Librería""",3500,4,14000
"""Lapicera""","""Librería""",1200,10,12000
"""Mochila""","""Accesorios""",28500,2,57000
"""Calculadora""","""Tecnología""",18500,3,55500
"""Regla""","""Librería""",900,6,5400
"""Carpeta""","""Librería""",4200,5,21000


La salida tiene ahora `shape: (6, 5)`. Se conservan `producto`, `categoria`, `precio` y `cantidad`, y se agrega `importe` como quinta columna.

La diferencia entre ambos contextos queda visible en la ejecución:

- `select()` construye el resultado solo con las expresiones indicadas;
- `with_columns()` conserva las columnas existentes y agrega o reemplaza las que producen las nuevas expresiones.


## 4. Guardar el resultado transformado

Como las operaciones anteriores, `with_columns()` devuelve un nuevo DataFrame. Podemos almacenarlo en otra variable.


In [25]:
ventas = df.with_columns(
    (pl.col("precio") * pl.col("cantidad")).alias("importe")
)

ventas

producto,categoria,precio,cantidad,importe
str,str,i64,i64,i64
"""Cuaderno""","""Librería""",3500,4,14000
"""Lapicera""","""Librería""",1200,10,12000
"""Mochila""","""Accesorios""",28500,2,57000
"""Calculadora""","""Tecnología""",18500,3,55500
"""Regla""","""Librería""",900,6,5400
"""Carpeta""","""Librería""",4200,5,21000


La variable `ventas` tiene `shape: (6, 5)` y contiene la nueva columna `importe` junto con las cuatro originales.


In [26]:
df

producto,categoria,precio,cantidad
str,str,i64,i64
"""Cuaderno""","""Librería""",3500,4
"""Lapicera""","""Librería""",1200,10
"""Mochila""","""Accesorios""",28500,2
"""Calculadora""","""Tecnología""",18500,3
"""Regla""","""Librería""",900,6
"""Carpeta""","""Librería""",4200,5


Al volver a mostrar `df`, vemos nuevamente `shape: (6, 4)`. La columna `importe` no está presente, lo que confirma que la llamada anterior a `with_columns()` produjo un nuevo DataFrame y no modificó el objeto original.


## 5. Crear varias columnas en una sola operación

Podemos proporcionar más de una expresión dentro de `with_columns()`.


In [27]:
df.with_columns(
    (pl.col("precio") * pl.col("cantidad")).alias("importe"),
    (pl.col("precio") * 1.10).alias("precio_con_aumento"),
)

producto,categoria,precio,cantidad,importe,precio_con_aumento
str,str,i64,i64,i64,f64
"""Cuaderno""","""Librería""",3500,4,14000,3850.0
"""Lapicera""","""Librería""",1200,10,12000,1320.0
"""Mochila""","""Accesorios""",28500,2,57000,31350.0
"""Calculadora""","""Tecnología""",18500,3,55500,20350.0
"""Regla""","""Librería""",900,6,5400,990.0
"""Carpeta""","""Librería""",4200,5,21000,4620.0


La salida tiene `shape: (6, 6)` y agrega dos columnas: `importe` y `precio_con_aumento`.

`importe` conserva tipo `Int64`, mientras que `precio_con_aumento` aparece como `Float64`. Los valores aumentados son `3850.0`, `1320.0`, `31350.0`, `20350.0`, `990.0` y `4620.0`.

Las dos expresiones se evaluaron a partir de las columnas que ya existían en `df`.


## 6. Utilizar argumentos con nombre

También podemos dar nombre a las columnas nuevas directamente dentro de `with_columns()`.


In [28]:
df.with_columns(
    importe=pl.col("precio") * pl.col("cantidad"),
    precio_en_miles=pl.col("precio") / 1000,
)

producto,categoria,precio,cantidad,importe,precio_en_miles
str,str,i64,i64,i64,f64
"""Cuaderno""","""Librería""",3500,4,14000,3.5
"""Lapicera""","""Librería""",1200,10,12000,1.2
"""Mochila""","""Accesorios""",28500,2,57000,28.5
"""Calculadora""","""Tecnología""",18500,3,55500,18.5
"""Regla""","""Librería""",900,6,5400,0.9
"""Carpeta""","""Librería""",4200,5,21000,4.2


La salida vuelve a tener seis columnas. `importe` contiene los mismos importes calculados anteriormente y `precio_en_miles` muestra `3.5`, `1.2`, `28.5`, `18.5`, `0.9` y `4.2`.

Los nombres `importe` y `precio_en_miles` se utilizaron directamente como nombres de las nuevas columnas. Esta sintaxis resulta especialmente clara cuando creamos varias variables con nombres definidos.


## 7. Crear una columna constante

No todas las columnas nuevas tienen que depender de otras variables. Podemos agregar un mismo valor a todas las filas mediante `pl.lit()`.


In [29]:
df.with_columns(
    moneda=pl.lit("ARS")
)

producto,categoria,precio,cantidad,moneda
str,str,i64,i64,str
"""Cuaderno""","""Librería""",3500,4,"""ARS"""
"""Lapicera""","""Librería""",1200,10,"""ARS"""
"""Mochila""","""Accesorios""",28500,2,"""ARS"""
"""Calculadora""","""Tecnología""",18500,3,"""ARS"""
"""Regla""","""Librería""",900,6,"""ARS"""
"""Carpeta""","""Librería""",4200,5,"""ARS"""


La salida tiene `shape: (6, 5)` y la nueva columna `moneda` contiene `"ARS"` en las seis filas.

Aquí `pl.lit("ARS")` representa un valor literal que Polars repite para construir la columna completa.


## 8. Encadenar operaciones dentro de una expresión

Podemos aplicar varias operaciones sobre una misma columna antes de incorporarla al DataFrame.


In [30]:
df.with_columns(
    precio_en_miles=(
        (pl.col("precio") / 1000)
        .round(2)
    )
)

producto,categoria,precio,cantidad,precio_en_miles
str,str,i64,i64,f64
"""Cuaderno""","""Librería""",3500,4,3.5
"""Lapicera""","""Librería""",1200,10,1.2
"""Mochila""","""Accesorios""",28500,2,28.5
"""Calculadora""","""Tecnología""",18500,3,18.5
"""Regla""","""Librería""",900,6,0.9
"""Carpeta""","""Librería""",4200,5,4.2


La nueva columna `precio_en_miles` contiene `3.5`, `1.2`, `28.5`, `18.5`, `0.9` y `4.2`. En este conjunto de datos esos valores ya tienen como máximo un decimal, por lo que el redondeo a dos decimales no modifica su representación visible.

La expresión sigue siendo útil para observar el encadenamiento: primero dividimos `precio` por `1000` y después aplicamos `.round(2)`.


## 9. Observar un cambio de tipo

El DataFrame original almacena `precio` como entero. Al dividirlo por `1000`, la columna resultante contiene valores decimales.

Podemos observar el esquema de la tabla transformada.


In [31]:
df.with_columns(
    precio_en_miles=pl.col("precio") / 1000
).schema

Schema([('producto', String),
        ('categoria', String),
        ('precio', Int64),
        ('cantidad', Int64),
        ('precio_en_miles', Float64)])

El esquema confirma el cambio de tipo:

```text
precio           → Int64
precio_en_miles  → Float64
```

Aunque `precio` contiene enteros, la división por `1000` produce una columna de punto flotante. Una transformación puede cambiar no solo los valores, sino también el tipo del resultado.


## 10. Reemplazar una columna existente

Si una expresión produce una columna con el mismo nombre que una columna ya existente, `with_columns()` reemplaza esa columna en el resultado.


In [32]:
df.with_columns(
    (pl.col("precio") * 1.10).alias("precio")
)

producto,categoria,precio,cantidad
str,str,f64,i64
"""Cuaderno""","""Librería""",3850.0,4
"""Lapicera""","""Librería""",1320.0,10
"""Mochila""","""Accesorios""",31350.0,2
"""Calculadora""","""Tecnología""",20350.0,3
"""Regla""","""Librería""",990.0,6
"""Carpeta""","""Librería""",4620.0,5


La salida conserva `shape: (6, 4)`, pero `precio` fue reemplazada por la versión aumentada en un 10 %. Sus valores pasan a `3850.0`, `1320.0`, `31350.0`, `20350.0`, `990.0` y `4620.0`.

También cambia su tipo: `precio` pasa de `Int64` a `Float64`. Como la expresión produjo una columna con el mismo nombre, `with_columns()` sustituyó la columna original en el resultado.


## 11. Comparar conservar y reemplazar

Podemos conservar simultáneamente el valor original y el transformado utilizando un nombre nuevo.


In [33]:
df.with_columns(
    precio_aumentado=pl.col("precio") * 1.10
).select(
    "producto",
    "precio",
    "precio_aumentado",
)

producto,precio,precio_aumentado
str,i64,f64
"""Cuaderno""",3500,3850.0
"""Lapicera""",1200,1320.0
"""Mochila""",28500,31350.0
"""Calculadora""",18500,20350.0
"""Regla""",900,990.0
"""Carpeta""",4200,4620.0


La tabla resultante tiene `shape: (6, 3)` y permite comparar directamente `precio` con `precio_aumentado`.

Por ejemplo, `Cuaderno` pasa de `3500` a `3850.0`, `Mochila` de `28500` a `31350.0` y `Carpeta` de `4200` a `4620.0`. Al utilizar un nombre nuevo conservamos simultáneamente el valor original y el transformado.


## 12. Reemplazar varias columnas a la vez

También podemos transformar varias columnas existentes dentro de una misma llamada.


In [34]:
df.with_columns(
    (pl.col("precio") * 1.10).alias("precio"),
    (pl.col("cantidad") * 2).alias("cantidad"),
)

producto,categoria,precio,cantidad
str,str,f64,i64
"""Cuaderno""","""Librería""",3850.0,8
"""Lapicera""","""Librería""",1320.0,20
"""Mochila""","""Accesorios""",31350.0,4
"""Calculadora""","""Tecnología""",20350.0,6
"""Regla""","""Librería""",990.0,12
"""Carpeta""","""Librería""",4620.0,10


La salida sigue teniendo cuatro columnas, pero dos de ellas fueron reemplazadas. `precio` aumenta un 10 % y pasa a `Float64`, mientras que `cantidad` duplica sus valores: `4` pasa a `8`, `10` a `20`, `2` a `4`, y así sucesivamente.

`producto` y `categoria` permanecen sin cambios.


## 13. Una dependencia entre columnas nuevas

Supongamos que queremos crear primero `importe` y después calcular `importe_con_impuesto` a partir de esa nueva columna.

No debemos pensar las expresiones de una misma llamada a `with_columns()` como instrucciones que se ejecutan una detrás de otra. Las expresiones parten del DataFrame que entra en ese contexto.

Por eso resulta más claro separar las dos etapas.


In [35]:
df.with_columns(
    importe=pl.col("precio") * pl.col("cantidad")
).with_columns(
    importe_con_impuesto=pl.col("importe") * 1.21
)

producto,categoria,precio,cantidad,importe,importe_con_impuesto
str,str,i64,i64,i64,f64
"""Cuaderno""","""Librería""",3500,4,14000,16940.0
"""Lapicera""","""Librería""",1200,10,12000,14520.0
"""Mochila""","""Accesorios""",28500,2,57000,68970.0
"""Calculadora""","""Tecnología""",18500,3,55500,67155.0
"""Regla""","""Librería""",900,6,5400,6534.0
"""Carpeta""","""Librería""",4200,5,21000,25410.0


La salida tiene `shape: (6, 6)` y muestra con claridad las dos etapas. La primera llamada crea `importe`; la segunda puede utilizar esa nueva columna para calcular `importe_con_impuesto`.

Por ejemplo, para `Cuaderno`, `importe` es `14000` y `importe_con_impuesto` es `16940.0`. Para `Mochila`, los valores son `57000` y `68970.0`.

El encadenamiento hace explícita la dependencia entre ambas transformaciones.


## 14. Dos columnas independientes en un mismo contexto

Si ambas columnas nuevas dependen solamente de variables que ya existían en `df`, sí podemos crearlas dentro de una sola llamada.


In [36]:
df.with_columns(
    importe=pl.col("precio") * pl.col("cantidad"),
    importe_con_impuesto=(
        pl.col("precio") * pl.col("cantidad") * 1.21
    ),
)

producto,categoria,precio,cantidad,importe,importe_con_impuesto
str,str,i64,i64,i64,f64
"""Cuaderno""","""Librería""",3500,4,14000,16940.0
"""Lapicera""","""Librería""",1200,10,12000,14520.0
"""Mochila""","""Accesorios""",28500,2,57000,68970.0
"""Calculadora""","""Tecnología""",18500,3,55500,67155.0
"""Regla""","""Librería""",900,6,5400,6534.0
"""Carpeta""","""Librería""",4200,5,21000,25410.0


La salida coincide con la anterior: `importe` e `importe_con_impuesto` contienen los mismos valores.

La diferencia está en cómo fueron construidas. Aquí ambas expresiones parten directamente de `precio` y `cantidad`, que ya existían en `df`, por lo que pueden evaluarse dentro de una misma llamada a `with_columns()`.


## 15. Combinar `filter()`, `with_columns()` y `select()`

Ya conocemos tres contextos diferentes y podemos utilizarlos dentro de una misma cadena.

Primero conservamos los productos cuyo precio supera `2000`, luego calculamos `importe` y finalmente seleccionamos las columnas que queremos mostrar.


In [37]:
df.filter(
    pl.col("precio") > 2000
).with_columns(
    importe=pl.col("precio") * pl.col("cantidad")
).select(
    "producto",
    "precio",
    "cantidad",
    "importe",
)

producto,precio,cantidad,importe
str,i64,i64,i64
"""Cuaderno""",3500,4,14000
"""Mochila""",28500,2,57000
"""Calculadora""",18500,3,55500
"""Carpeta""",4200,5,21000


La salida final tiene `shape: (4, 4)`. Después del filtro permanecen `Cuaderno`, `Mochila`, `Calculadora` y `Carpeta`, porque sus precios superan `2000`.

Sobre esas cuatro filas se calcula `importe`, obteniendo `14000`, `57000`, `55500` y `21000`, y `select()` deja finalmente las columnas `producto`, `precio`, `cantidad` e `importe`.

La cadena se lee de izquierda a derecha:

1. conservar las filas con `precio > 2000`;
2. agregar `importe`;
3. construir el resultado final con las cuatro columnas elegidas.


## 16. Reasignar una transformación

Cuando queremos continuar trabajando con el DataFrame transformado, podemos reasignar el resultado al mismo nombre.


In [38]:
df_transformado = df.with_columns(
    importe=pl.col("precio") * pl.col("cantidad"),
    moneda=pl.lit("ARS"),
)

df_transformado

producto,categoria,precio,cantidad,importe,moneda
str,str,i64,i64,i64,str
"""Cuaderno""","""Librería""",3500,4,14000,"""ARS"""
"""Lapicera""","""Librería""",1200,10,12000,"""ARS"""
"""Mochila""","""Accesorios""",28500,2,57000,"""ARS"""
"""Calculadora""","""Tecnología""",18500,3,55500,"""ARS"""
"""Regla""","""Librería""",900,6,5400,"""ARS"""
"""Carpeta""","""Librería""",4200,5,21000,"""ARS"""


`df_transformado` tiene `shape: (6, 6)` y conserva las cuatro columnas originales junto con `importe` y `moneda`.

`importe` contiene `14000`, `12000`, `57000`, `55500`, `5400` y `21000`, mientras que `moneda` contiene `"ARS"` en todas las filas. `df` no fue modificado; simplemente almacenamos el resultado en una variable nueva.


## 17. Qué conviene recordar

`with_columns()` es el contexto que utilizaremos habitualmente cuando queramos conservar un DataFrame y **agregar o reemplazar columnas mediante expresiones**.

En este cuaderno vimos que podemos:

- crear una columna calculada;
- agregar varias columnas en una sola llamada;
- utilizar `alias()` o argumentos con nombre;
- incorporar valores constantes con `pl.lit()`;
- encadenar operaciones dentro de una expresión;
- reemplazar columnas existentes;
- observar cambios de tipo;
- encadenar varios `with_columns()` cuando existe una dependencia;
- combinar `filter()`, `with_columns()` y `select()`.

El patrón general continúa siendo el mismo: describimos transformaciones sobre columnas completas y Polars construye un nuevo DataFrame con el resultado.


## Próximo paso

Hasta aquí nuestras transformaciones aplicaron la misma regla a todas las filas.

En el próximo capítulo resolveremos situaciones en las que el valor producido dependa de una condición. Por ejemplo:

```text
si precio > 10000
    "Alto"
en caso contrario
    "Normal"
```

Para eso utilizaremos `when()`, `then()` y `otherwise()`, integrando la lógica booleana que ya conocemos con la creación de columnas.


---

### Autoría y atribución

Este cuaderno forma parte del material educativo desarrollado por **VintaBytes**.

📧 **Contacto:** [vintabytes@gmail.com](mailto:vintabytes@gmail.com)

🌐 **Repositorio oficial:** [github.com/VintaBytes/Ciencia-de-datos](https://github.com/VintaBytes/Ciencia-de-datos)

Se permite compartir y reutilizar este material respetando la autoría y manteniendo visible esta atribución.

---
